In [6]:
import pandas as pd
import json
from scipy.stats import chi2
# To parse the initialization events, group the assignments by device and browser, 
# and highlight any segments that heavily deviate from a 50/50 split.
# 1. Load Data
events = pd.read_csv('ab_hero_v4_events.csv')
valid_events = events[events['session_traffic_quality'] == 'valid'].copy()

# 2. Extract Arm Assignment cleanly
def get_arm(event_data):
    try:
        if pd.isna(event_data):
            return None
        return json.loads(event_data).get('experiment_var')
    except Exception:
        return None

ab_init = valid_events[valid_events['event_name'] == 'ab_experiment_init'].copy()
ab_init['arm'] = ab_init['event_data'].apply(get_arm)

# Deduplicate to get one arm per user
user_arm = (
    ab_init.sort_values('ingestion_timestamp')
    .groupby('client_id')
    .first()
    .reset_index()
)

# 3. Analyze SRM by Segment (Device & Browser)
def check_srm(df, segment_col):
    print(f"\n--- SRM Check by {segment_col.upper()} ---")
    segments = df[segment_col].dropna().unique()
    
    for seg in segments:
        seg_data = df[df[segment_col] == seg]
        a_count = (seg_data['arm'] == 'a').sum()
        b_count = (seg_data['arm'] == 'b').sum()
        total = a_count + b_count
        
        if total == 0:
            continue
            
        expected = total / 2
        chi2_val = ((a_count - expected)**2 + (b_count - expected)**2) / expected
        p_val = 1 - chi2.cdf(chi2_val, df=1)   # ✅ FIX: use keyword 'df' correctly
        
        split_a = (a_count / total) * 100
        split_b = (b_count / total) * 100
        
        # Flag highly significant mismatches
        flag = " ⚠️ FLAG" if p_val < 0.05 else ""
        print(f"{seg}: Total={total} | A:B = {split_a:.1f}%:{split_b:.1f}% | p-val={p_val:.4f}{flag}")

# Run SRM checks
check_srm(user_arm, 'device')
check_srm(user_arm, 'browser')



--- SRM Check by DEVICE ---
Mobile: Total=3894 | A:B = 48.9%:51.1% | p-val=0.1783
Desktop: Total=992 | A:B = 46.1%:53.9% | p-val=0.0133 ⚠️ FLAG
Tablet: Total=347 | A:B = 38.9%:61.1% | p-val=0.0000 ⚠️ FLAG

--- SRM Check by BROWSER ---
Chrome: Total=1439 | A:B = 50.5%:49.5% | p-val=0.7318
Safari: Total=1956 | A:B = 48.2%:51.8% | p-val=0.1035
Edge: Total=228 | A:B = 43.9%:56.1% | p-val=0.0637
Facebook: Total=1176 | A:B = 45.1%:54.9% | p-val=0.0007 ⚠️ FLAG
Firefox: Total=80 | A:B = 43.8%:56.2% | p-val=0.2636
Samsung Internet: Total=44 | A:B = 47.7%:52.3% | p-val=0.7630
Instagram: Total=299 | A:B = 45.8%:54.2% | p-val=0.1482
Other: Total=2 | A:B = 50.0%:50.0% | p-val=1.0000
Opera: Total=9 | A:B = 55.6%:44.4% | p-val=0.7389


In [8]:
import pandas as pd
import json
import numpy as np
# To extract the median physical location of the size selection clicks across both arms.
# 1. Isolate clicks from valid events
clicks = valid_events[valid_events['event_name'] == 'clicked'].copy()

# 2. Parse the event_data JSON for clicks safely
def parse_click(data_str):
    try:
        if pd.isna(data_str): return pd.Series([None, None, np.nan])
        d = json.loads(data_str)
        coords = d.get('click_coordinates')
        y = np.nan
        if isinstance(coords, dict):
            y = coords.get('y')
        elif isinstance(coords, list) and len(coords) >= 2:
            y = coords[1]
        return pd.Series([d.get('element_id'), d.get('element_text'), float(y) if y is not None else np.nan])
    except:
        return pd.Series([None, None, np.nan])

# Apply and assign new columns
clicks[['element_id', 'element_text', 'y_coord']] = clicks['event_data'].apply(parse_click)

# 3. Merge ONLY the arm assignment
clicks_arm = clicks.merge(user_arm[['client_id', 'arm']], on='client_id', how='inner')
clicks_arm = clicks_arm.dropna(subset=['y_coord'])

# 4. Filter for size selector clicks
size_keywords = 'Twin|Full|Queen|King|California King'
size_clicks = clicks_arm[clicks_arm['element_text'].astype(str).str.contains(size_keywords, case=False, na=False)]

# 5. Group and aggregate
summary = size_clicks.groupby(['arm', 'device']).agg(
    Median_Y_Pixel_Depth=('y_coord', 'median'),
    Total_Clicks=('y_coord', 'count')
).reset_index()

print("\n--- Physical Location of Size Selector Clicks (Y-Axis) ---")
print(summary.to_string(index=False))


--- Physical Location of Size Selector Clicks (Y-Axis) ---
arm  device  Median_Y_Pixel_Depth  Total_Clicks
  a Desktop                 474.0          1273
  a  Mobile                 457.0          3572
  a  Tablet                 488.0           225
  b Desktop                 500.0          1525
  b  Mobile                 408.0          4119
  b  Tablet                 534.5           262


In [9]:
import pandas as pd
# To calculate the Time to First Interaction (TTFI). We will measure the exact time delta in seconds between the 
# page_viewed event and the user's very next event, grouped by arm and device.
# 1. Ensure timestamps are datetime objects
valid_events['ingestion_timestamp'] = pd.to_datetime(valid_events['ingestion_timestamp'])

# 2. Sort chronologically per user and session
sorted_events = valid_events.sort_values(['client_id', 'session_id', 'ingestion_timestamp'])

# 3. Shift the column to get the timestamp of the NEXT event in the session
sorted_events['next_event_time'] = sorted_events.groupby(['client_id', 'session_id'])['ingestion_timestamp'].shift(-1)

# 4. Filter for the start of the session (page_viewed)
pv_events = sorted_events[sorted_events['event_name'] == 'page_viewed'].copy()

# 5. Calculate the time delta in seconds
pv_events['time_to_interact_sec'] = (pv_events['next_event_time'] - pv_events['ingestion_timestamp']).dt.total_seconds()

# 6. Drop bounces (users who had no second event) and merge arm/device data
ttfi = pv_events.dropna(subset=['time_to_interact_sec'])
ttfi = ttfi.merge(user_arm[['client_id', 'arm']], on='client_id', how='inner')

# 7. Aggregate median time to first interaction
summary_ttfi = ttfi.groupby(['arm', 'device'])['time_to_interact_sec'].median().reset_index()

print("\n--- Median Time to First Interaction (Seconds) ---")
print(summary_ttfi.to_string(index=False))


--- Median Time to First Interaction (Seconds) ---
arm  device  time_to_interact_sec
  a Desktop                   2.0
  a  Mobile                   2.0
  a  Tablet                   3.0
  b Desktop                   1.0
  b  Mobile                   1.0
  b  Tablet                   1.0


In [13]:
import pandas as pd
import json
import numpy as np
from scipy.stats import chi2, norm

# ==========================================
# 1. LOAD DATA & FILTER VALID TRAFFIC
# ==========================================
events = pd.read_csv('ab_hero_v4_events.csv')
valid_events = events[events['session_traffic_quality'] == 'valid'].copy()

# ==========================================
# 2. ARM ASSIGNMENT
# ==========================================
def get_arm(event_data):
    try:
        if pd.isna(event_data): return None
        return json.loads(event_data).get('experiment_var')
    except:
        return None

ab_init = valid_events[valid_events['event_name'] == 'ab_experiment_init'].copy()
ab_init['arm'] = ab_init['event_data'].apply(get_arm)
user_arm = ab_init.sort_values('ingestion_timestamp').groupby('client_id').first().reset_index()

# Merge valid events with their definitive arm assignment
valid = valid_events.merge(user_arm[['client_id', 'arm']], on='client_id', how='inner')

# ==========================================
# 3. MACRO FUNNEL & DEVICE SEGMENTATION
# ==========================================
funnel_steps = [
    'page_viewed', 'size_changed', 'product_added_to_cart', 
    'checkout_initiated', 'checkout_started', 'checkout_completed'
]

print("--- 1. Macro Funnel (Valid Traffic) ---")
for arm in ['a', 'b']:
    arm_data = valid[valid['arm'] == arm]
    total = arm_data['client_id'].nunique()
    print(f"\nArm {arm.upper()} (Users: {total}):")
    for event in funnel_steps:
        n = arm_data[arm_data['event_name'] == event]['client_id'].nunique()
        print(f"  {event}: {n} ({(n/total)*100:.1f}%)")

print("\n--- 2. Device Segmentation (Page View -> Size Changed) ---")
for arm in ['a', 'b']:
    arm_data = valid[valid['arm'] == arm]
    devices = arm_data['device'].dropna().unique()
    print(f"\nArm {arm.upper()}:")
    for dev in devices:
        dev_data = arm_data[arm_data['device'] == dev]
        pv = dev_data[dev_data['event_name'] == 'page_viewed']['client_id'].nunique()
        sc = dev_data[dev_data['event_name'] == 'size_changed']['client_id'].nunique()
        if pv > 0:
            print(f"  {dev}: {sc}/{pv} ({(sc/pv)*100:.1f}%)")

# ==========================================
# 4. SRM CHECK (ISOLATING THE BUCKETING BUG)
# ==========================================
def check_srm(df, segment_col):
    print(f"\n--- 3. SRM Check by {segment_col.upper()} ---")
    segments = df[segment_col].dropna().unique()
    for seg in segments:
        seg_data = df[df[segment_col] == seg]
        a_count = len(seg_data[seg_data['arm'] == 'a'])
        b_count = len(seg_data[seg_data['arm'] == 'b'])
        total = a_count + b_count
        if total == 0: continue
        
        expected = total / 2
        chi2_val = ((a_count - expected)**2 + (b_count - expected)**2) / expected
        p_val = 1 - chi2.cdf(chi2_val, df=1)
        
        flag = " ⚠️ FLAG (Significant)" if p_val < 0.05 else ""
        print(f"  {seg}: Total={total} | A:B = {(a_count/total)*100:.1f}%:{(b_count/total)*100:.1f}% | p-val={p_val:.4f}{flag}")

check_srm(user_arm, 'device')
check_srm(user_arm, 'browser')

# ==========================================
# 5. PHYSICAL REACHABILITY (Y-Axis Clicks)
# ==========================================
clicks = valid[valid['event_name'] == 'clicked'].copy()

def parse_click(data_str):
    try:
        if pd.isna(data_str): return pd.Series([None, np.nan])
        d = json.loads(data_str)
        coords = d.get('click_coordinates')
        y = np.nan
        if isinstance(coords, dict): y = coords.get('y')
        elif isinstance(coords, list) and len(coords) >= 2: y = coords[1]
        return pd.Series([d.get('element_text'), float(y) if y is not None else np.nan])
    except:
        return pd.Series([None, np.nan])

clicks[['element_text', 'y_coord']] = clicks['event_data'].apply(parse_click)
clicks = clicks.dropna(subset=['y_coord'])
size_clicks = clicks[clicks['element_text'].astype(str).str.contains('Twin|Full|Queen|King', case=False, na=False)]

print("\n--- 4. Physical Reachability (Median Y-Axis Pixel Depth) ---")
y_summary = size_clicks.groupby(['arm', 'device'])['y_coord'].median().reset_index()
print(y_summary.to_string(index=False))

# ==========================================
# 6. LATENCY (Time to First Interaction)
# ==========================================
valid['ingestion_timestamp'] = pd.to_datetime(valid['ingestion_timestamp'])
sorted_events = valid.sort_values(['client_id', 'session_id', 'ingestion_timestamp'])
sorted_events['next_event_time'] = sorted_events.groupby(['client_id', 'session_id'])['ingestion_timestamp'].shift(-1)

pv_events = sorted_events[sorted_events['event_name'] == 'page_viewed'].copy()
pv_events['ttfi_sec'] = (pv_events['next_event_time'] - pv_events['ingestion_timestamp']).dt.total_seconds()
ttfi = pv_events.dropna(subset=['ttfi_sec'])

print("\n--- 5. Median Time to First Interaction (Seconds) ---")
ttfi_summary = ttfi.groupby(['arm', 'device'])['ttfi_sec'].median().reset_index()
print(ttfi_summary.to_string(index=False))

# ==========================================
# 7. MACRO REVENUE & CONDITIONAL INTENT (Z-TEST)
# ==========================================
orders = pd.read_csv('ab_hero_v4_order_line_items.csv')
cc_events = valid[valid['event_name'] == 'checkout_completed'].copy()

def get_order_id(event_data):
    try:
        if pd.isna(event_data): return None
        return json.loads(event_data).get('order_id')
    except:
        return None

cc_events['order_id_parsed'] = cc_events['event_data'].apply(get_order_id)
cc_events['order_id_parsed'] = pd.to_numeric(cc_events['order_id_parsed'], errors='coerce')

orders_joined = orders.merge(
    cc_events[['order_id_parsed', 'arm', 'client_id']].drop_duplicates(),
    left_on='order_id', right_on='order_id_parsed', how='inner'
)
order_totals = orders_joined.groupby(['order_id', 'arm'])['value'].sum().reset_index()

print("\n--- 6. Revenue Verification ---")
for arm in ['a', 'b']:
    ao = order_totals[order_totals['arm'] == arm]
    print(f"Arm {arm.upper()}: {len(ao)} orders | Total Rev: ${ao['value'].sum():,.0f}")

# Conditional ATC Z-Test
print("\n--- 7. Conditional ATC Z-Test ---")
cond_stats = {}
for arm in ['a', 'b']:
    arm_data = valid[valid['arm'] == arm]
    sc = arm_data[arm_data['event_name'] == 'size_changed']['client_id'].nunique()
    atc = arm_data[arm_data['event_name'] == 'product_added_to_cart']['client_id'].nunique()
    if sc > 0:
        cond_stats[arm] = {'success': atc, 'total': sc}

p1 = cond_stats['a']['success'] / cond_stats['a']['total']
p2 = cond_stats['b']['success'] / cond_stats['b']['total']
n1 = cond_stats['a']['total']
n2 = cond_stats['b']['total']

p_pool = (cond_stats['a']['success'] + cond_stats['b']['success']) / (n1 + n2)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
z = (p1 - p2) / se
p_val = 2 * (1 - norm.cdf(abs(z)))

print(f"Arm A Conditional ATC: {p1*100:.1f}%")
print(f"Arm B Conditional ATC: {p2*100:.1f}%")
print(f"Z-Score: {z:.3f} | p-value: {p_val:.3f}")

--- 1. Macro Funnel (Valid Traffic) ---

Arm A (Users: 2497):
  page_viewed: 2497 (100.0%)
  size_changed: 2079 (83.3%)
  product_added_to_cart: 476 (19.1%)
  checkout_initiated: 195 (7.8%)
  checkout_started: 130 (5.2%)
  checkout_completed: 66 (2.6%)

Arm B (Users: 2736):
  page_viewed: 2736 (100.0%)
  size_changed: 1980 (72.4%)
  product_added_to_cart: 491 (17.9%)
  checkout_initiated: 158 (5.8%)
  checkout_started: 108 (3.9%)
  checkout_completed: 49 (1.8%)

--- 2. Device Segmentation (Page View -> Size Changed) ---

Arm A:
  Mobile: 1568/1905 (82.3%)
  Desktop: 413/457 (90.4%)
  Tablet: 98/135 (72.6%)

Arm B:
  Mobile: 1390/1989 (69.9%)
  Desktop: 452/535 (84.5%)
  Tablet: 138/212 (65.1%)

--- 3. SRM Check by DEVICE ---
  Mobile: Total=3894 | A:B = 48.9%:51.1% | p-val=0.1783
  Desktop: Total=992 | A:B = 46.1%:53.9% | p-val=0.0133 ⚠️ FLAG (Significant)
  Tablet: Total=347 | A:B = 38.9%:61.1% | p-val=0.0000 ⚠️ FLAG (Significant)

--- 3. SRM Check by BROWSER ---
  Chrome: Total=1439 